# 02 - Data Preparation

## هدف

در این نوت‌بوک، داده‌های بریتانیا و فرانسه برای مدل‌سازی بین‌کشوری آماده می‌شوند.

مراحل اصلی شامل:
- انتخاب متغیرهای موردنیاز
- مدیریت مقادیر نامعتبر یا ناشناخته
- هماهنگ‌سازی کدگذاری متغیرهای مشترک
- ایجاد ساختار یکسان برای دو کشور
- ذخیره نسخه‌های پردازش‌شده

اصل این مرحله، حفظ سادگی و قابلیت بازتولید است. از پاک‌سازی و مهندسی ویژگی غیرضروری اجتناب می‌شود

In [ ]:
# وارد کردن کتابخانه‌های موردنیاز برای پردازش و مدیریت داده‌ها
import pandas as pd
import numpy as np

In [ ]:
# تعریف مسیر نسبی فایل‌های خام UK و France برای اجرای مستقل و سازگار با GitHub
# UK
uk_collisions_path = "../data/raw/UK/dft-road-casualty-statistics-collision-2025.csv"

# France
fr_characteristics_path = "../data/raw/France/caract-2024.csv"
fr_locations_path = "../data/raw/France/lieux-2024.csv"
fr_users_path = "../data/raw/France/usagers-2024.csv"

In [ ]:
# بارگذاری جداول موردنیاز و بررسی ابعاد اولیه آن‌ها
uk_collisions = pd.read_csv(uk_collisions_path)

fr_characteristics = pd.read_csv(fr_characteristics_path, sep=";")
fr_locations = pd.read_csv(fr_locations_path, sep=";")
fr_users = pd.read_csv(fr_users_path, sep=";")

print("UK Collisions:", uk_collisions.shape)
print("France Characteristics:", fr_characteristics.shape)
print("France Locations:", fr_locations.shape)
print("France Users:", fr_users.shape)

In [ ]:
# تبدیل کدهای شدت تصادف UK به سه کلاس Fatal، Serious و Slight

uk_severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

uk_collisions["severity_class"] = (
    uk_collisions["collision_severity"]
    .map(uk_severity_map)
)

In [ ]:
# تبدیل شدت افراد France به شدت هر تصادف براساس شدیدترین پیامد ثبت‌شده

severity_rank_fr = {
    1: 0,  # Uninjured
    4: 1,  # Slight
    3: 2,  # Hospitalised
    2: 3   # Killed
}

fr_users["severity_rank"] = fr_users["grav"].map(severity_rank_fr)

fr_collision_severity = (
    fr_users.groupby("Num_Acc")["severity_rank"]
    .max()
    .reset_index()
)

fr_severity_map = {
    1: "Slight",
    2: "Serious",
    3: "Fatal"
}

fr_collision_severity["severity_class"] = (
    fr_collision_severity["severity_rank"]
    .map(fr_severity_map)
)

In [ ]:
# بررسی توزیع کلاس‌های شدت پس از هماهنگ‌سازی متغیر هدف

print("UK severity:")
print(uk_collisions["severity_class"].value_counts())

print("\nFrance severity:")
print(fr_collision_severity["severity_class"].value_counts())

In [ ]:
# ساخت دیتاست پایه UK با متغیر هدف و ویژگی‌های منتخب در سطح تصادف

uk_base = uk_collisions[
    [
        "collision_index",
        "severity_class",
        "time",
        "light_conditions",
        "weather_conditions",
        "road_surface_conditions",
        "urban_or_rural_area",
        "speed_limit",
        "junction_detail",
        "first_road_class"
    ]
].copy()

print("UK base shape:", uk_base.shape)
uk_base.head()

In [ ]:
# بررسی تعداد رکوردهای Locations برای هر تصادف فرانسه پیش از ادغام

fr_location_counts = (
    fr_locations.groupby("Num_Acc")
    .size()
)

print("Accidents with one location row:",
      (fr_location_counts == 1).sum())

print("Accidents with multiple location rows:",
      (fr_location_counts > 1).sum())

print("Maximum location rows for one accident:",
      fr_location_counts.max())

In [ ]:
# مشاهده چند نمونه از تصادف‌هایی که بیش از یک ردیف در جدول Locations دارند

multi_location_ids = fr_location_counts[
    fr_location_counts > 1
].index[:5]

fr_locations[
    fr_locations["Num_Acc"].isin(multi_location_ids)
].sort_values("Num_Acc")

In [ ]:
# بررسی اختلاف ویژگی‌های منتخب در تصادف‌های دارای چند ردیف Locations

location_feature_variation = (
    fr_locations
    .groupby("Num_Acc")[
        ["catr", "surf", "vma"]
    ]
    .nunique()
)

print("Accidents with different road class:",
      (location_feature_variation["catr"] > 1).sum())

print("Accidents with different road surface:",
      (location_feature_variation["surf"] > 1).sum())

print("Accidents with different speed limit:",
      (location_feature_variation["vma"] > 1).sum())

In [ ]:
# بررسی تعداد تصادف‌های دارای بیش از یک مقدار برای ویژگی‌های منتخب Locations

print("Different road class:",
      (location_feature_variation["catr"] > 1).sum())

print("Different road surface:",
      (location_feature_variation["surf"] > 1).sum())

print("Different speed limit:",
      (location_feature_variation["vma"] > 1).sum())

print("\nPercentage with different road surface:",
      (location_feature_variation["surf"] > 1).sum()
      / fr_locations["Num_Acc"].nunique() * 100)

In [ ]:
# بارگذاری دیتاست Addis Ababa و بررسی ابعاد اولیه

ethiopia_path = "../data/raw/Ethiopia/Addis_Ababa_city_RTA.csv"

ethiopia = pd.read_csv(ethiopia_path)

print("Ethiopia shape:", ethiopia.shape)
print("Number of columns:", len(ethiopia.columns))

In [ ]:
# نمایش نام تمام فیلدهای دیتاست Addis Ababa

for i, col in enumerate(ethiopia.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# بررسی نوع داده ستون‌ها و مقادیر گمشده

print(ethiopia.dtypes)

print("\nMissing values:")
print(
    ethiopia.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# بررسی علل ثبت‌شده برای تصادفات در دیتاست Addis Ababa

print("Cause of accident:")
print(
    ethiopia["Cause_of_accident"]
    .value_counts(dropna=False)
)

print("\nNumber of unique causes:",
      ethiopia["Cause_of_accident"].nunique())

In [ ]:
# بررسی توزیع کلاس‌های شدت تصادف در دیتاست Addis Ababa

print(
    ethiopia["Accident_severity"]
    .value_counts(dropna=False)
)

print("\nPercent:")
print(
    ethiopia["Accident_severity"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

In [ ]:
# بررسی مقادیر ویژگی‌های اصلی راننده، جاده و محیط در دیتاست Ethiopia

features_to_check = [
    "Age_band_of_driver",
    "Drivers_gender",
    "Driving_experience",
    "Type_of_vehicle",
    "Types_of_Junction",
    "Road_surface_type",
    "Road_surface_conditions",
    "Light_conditions",
    "Weather_conditions"
]

for col in features_to_check:
    print(f"\n--- {col} ---")
    print(ethiopia[col].value_counts(dropna=False))

In [ ]:
# بررسی ساختار زمان و روز هفته در دیتاست Ethiopia

print("--- Time ---")
print(ethiopia["Time"].head(20))
print("Unique times:", ethiopia["Time"].nunique())

print("\n--- Day of week ---")
print(ethiopia["Day_of_week"].value_counts(dropna=False))

In [ ]:
# نمایش ستون‌های جدول Users فرانسه برای بررسی امکان استفاده از ویژگی‌های راننده

for i, col in enumerate(fr_users.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# بررسی دسته کاربران جاده و موقعیت افراد برای شناسایی رانندگان در داده France

print("--- catu ---")
print(fr_users["catu"].value_counts(dropna=False).sort_index())

print("\n--- place ---")
print(fr_users["place"].value_counts(dropna=False).sort_index())

print("\n--- sexe ---")
print(fr_users["sexe"].value_counts(dropna=False).sort_index())

print("\n--- an_nais ---")
print(fr_users["an_nais"].describe())
print("Missing birth year:", fr_users["an_nais"].isna().sum())

In [ ]:
# بررسی تعداد رانندگان ثبت‌شده برای هر تصادف در France

fr_drivers = fr_users[
    fr_users["catu"] == 1
].copy()

drivers_per_accident = (
    fr_drivers.groupby("Num_Acc")
    .size()
)

print("Total drivers:", len(fr_drivers))
print("Accidents with driver records:", drivers_per_accident.size)

print("\nDrivers per accident:")
print(drivers_per_accident.value_counts().sort_index())

print("\nMaximum drivers in one accident:",
      drivers_per_accident.max())

print("\nAccidents without a driver record:",
      fr_characteristics["Num_Acc"].nunique()
      - drivers_per_accident.size)

In [ ]:
# استخراج ساعت وقوع تصادف با قالب یکسان برای UK، France و Ethiopia

uk_collisions["hour"] = pd.to_datetime(
    uk_collisions["time"],
    format="%H:%M",
    errors="coerce"
).dt.hour

fr_characteristics["hour"] = pd.to_datetime(
    fr_characteristics["hrmn"],
    format="%H:%M",
    errors="coerce"
).dt.hour

ethiopia["hour"] = pd.to_datetime(
    ethiopia["Time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour

In [ ]:
# کنترل محدوده، مقادیر گمشده و تعداد ساعات موجود پس از هماهنگ‌سازی زمان

for name, df in {
    "UK": uk_collisions,
    "France": fr_characteristics,
    "Ethiopia": ethiopia
}.items():

    print(f"\n--- {name} ---")
    print("Min hour:", df["hour"].min())
    print("Max hour:", df["hour"].max())
    print("Missing:", df["hour"].isna().sum())
    print("Unique hours:", df["hour"].nunique())

In [ ]:
# بررسی مقادیر روز هفته در UK و Ethiopia پیش از هماهنگ‌سازی

print("--- UK day_of_week ---")
print(
    uk_collisions["day_of_week"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n--- Ethiopia Day_of_week ---")
print(
    ethiopia["Day_of_week"]
    .value_counts(dropna=False)
)

In [ ]:
# بررسی سال، ماه و روز تصادفات France پیش از ساخت تاریخ کامل

print("--- France year ---")
print(fr_characteristics["an"].value_counts(dropna=False).sort_index())

print("\n--- France month ---")
print(fr_characteristics["mois"].value_counts(dropna=False).sort_index())

print("\n--- France day ---")
print(fr_characteristics["jour"].describe())

print("\nMissing:")
print(
    fr_characteristics[
        ["an", "mois", "jour"]
    ].isna().sum()
)

In [ ]:
# هماهنگ‌سازی روز هفته در UK، France و Ethiopia به نام استاندارد انگلیسی

uk_day_map = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

uk_collisions["day_common"] = (
    uk_collisions["day_of_week"]
    .map(uk_day_map)
)

fr_characteristics["accident_date"] = pd.to_datetime(
    dict(
        year=fr_characteristics["an"],
        month=fr_characteristics["mois"],
        day=fr_characteristics["jour"]
    ),
    errors="coerce"
)

fr_characteristics["day_common"] = (
    fr_characteristics["accident_date"]
    .dt.day_name()
)

ethiopia["day_common"] = (
    ethiopia["Day_of_week"]
    .str.strip()
)

In [ ]:
# کنترل مقادیر، Missing و توزیع روز هفته پس از هماهنگ‌سازی سه کشور

for name, df in {
    "UK": uk_collisions,
    "France": fr_characteristics,
    "Ethiopia": ethiopia
}.items():

    print(f"\n--- {name} ---")

    print(
        df["day_common"]
        .value_counts(dropna=False)
    )

    print("Missing:",
          df["day_common"].isna().sum())

    print("Unique:",
          df["day_common"].nunique())

In [ ]:
# هماهنگ‌سازی شرایط آب‌وهوایی در UK، France و Ethiopia

uk_weather_map = {
    1: "Clear",
    2: "Rain",
    3: "Snow",
    4: "Wind",
    5: "Rain",
    6: "Snow",
    7: "Fog",
    8: "Other",
    9: "Unknown",
    -1: "Unknown"
}

fr_weather_map = {
    1: "Clear",
    2: "Rain",
    3: "Rain",
    4: "Snow",
    5: "Fog",
    6: "Wind",
    7: "Other",
    8: "Other",
    9: "Other",
    -1: "Unknown"
}

eth_weather_map = {
    "Normal": "Clear",
    "Raining": "Rain",
    "Raining and Windy": "Rain",
    "Snow": "Snow",
    "Fog or mist": "Fog",
    "Windy": "Wind",
    "Cloudy": "Other",
    "Other": "Other",
    "Unknown": "Unknown"
}

uk_collisions["weather_common"] = (
    uk_collisions["weather_conditions"]
    .map(uk_weather_map)
)

fr_characteristics["weather_common"] = (
    fr_characteristics["atm"]
    .map(fr_weather_map)
)

ethiopia["weather_common"] = (
    ethiopia["Weather_conditions"]
    .str.strip()
    .map(eth_weather_map)
)

for name, data in {
    "UK": uk_collisions,
    "France": fr_characteristics,
    "Ethiopia": ethiopia
}.items():
    
    print(f"\n--- {name} ---")
    print(data["weather_common"].value_counts(dropna=False))
    print("Missing:", data["weather_common"].isna().sum())

In [ ]:
# هماهنگ‌سازی شرایط روشنایی در UK، France و Ethiopia

uk_light_map = {
    1: "Daylight",
    4: "Dark_Lit",
    5: "Dark_Unlit",
    6: "Dark_NoLighting",
    7: "Dark_Unknown",
    -1: "Unknown"
}

fr_light_map = {
    1: "Daylight",
    2: "Twilight",
    3: "Dark_NoLighting",
    4: "Dark_Unlit",
    5: "Dark_Lit"
}

eth_light_map = {
    "Daylight": "Daylight",
    "Darkness - lights lit": "Dark_Lit",
    "Darkness - lights unlit": "Dark_Unlit",
    "Darkness - no lighting": "Dark_NoLighting"
}

uk_collisions["light_common"] = (
    uk_collisions["light_conditions"]
    .map(uk_light_map)
)

fr_characteristics["light_common"] = (
    fr_characteristics["lum"]
    .map(fr_light_map)
)

ethiopia["light_common"] = (
    ethiopia["Light_conditions"]
    .str.strip()
    .map(eth_light_map)
)

for name, data in {
    "UK": uk_collisions,
    "France": fr_characteristics,
    "Ethiopia": ethiopia
}.items():

    print(f"\n--- {name} ---")
    print(data["light_common"].value_counts(dropna=False))
    print("Missing:", data["light_common"].isna().sum())

In [ ]:
# Harmonize road surface conditions across UK, France, and Ethiopia

uk_surface_map = {
    1: "Dry",
    2: "Wet",
    3: "Snow",
    4: "Ice",
    5: "Flood",
    9: "Unknown",
    -1: "Unknown"
}

fr_surface_map = {
    1: "Dry",
    2: "Wet",
    3: "Flood",
    4: "Flood",
    5: "Snow",
    6: "Other",
    7: "Ice",
    8: "Other",
    9: "Other",
    -1: "Unknown"
}

eth_surface_map = {
    "Dry": "Dry",
    "Wet or damp": "Wet",
    "Snow": "Snow",
    "Flood over 3cm. deep": "Flood"
}


uk_collisions["surface_common"] = (
    uk_collisions["road_surface_conditions"]
    .map(uk_surface_map)
)


fr_locations["surface_common"] = (
    fr_locations["surf"]
    .map(fr_surface_map)
)


ethiopia["surface_common"] = (
    ethiopia["Road_surface_conditions"]
    .str.strip()
    .map(eth_surface_map)
)


for name, data in {
    "UK": uk_collisions,
    "France": fr_locations,
    "Ethiopia": ethiopia
}.items():

    print(f"\n--- {name} ---")
    print(data["surface_common"].value_counts(dropna=False))
    print("Missing:",
          data["surface_common"].isna().sum())

In [ ]:
# بررسی مقادیر خام نوع تقاطع در سه کشور قبل از هماهنگ‌سازی

print("--- UK junction_detail ---")
print(
    uk_collisions["junction_detail"]
    .value_counts(dropna=False)
)

print("\n--- France intersection ---")
print(
    fr_characteristics["int"]
    .value_counts(dropna=False)
)

print("\n--- Ethiopia junction ---")
print(
    ethiopia["Types_of_Junction"]
    .value_counts(dropna=False)
)

In [ ]:
# Harmonize junction characteristics across UK, France, and Ethiopia

uk_junction_map = {
    0: "No_Junction",
    13: "T_Junction",
    16: "Crossroad",
    17: "Roundabout",
    18: "Private_Access",
    19: "Other",
    99: "Unknown",
    -1: "Unknown"
}


fr_junction_map = {
    1: "No_Junction",
    2: "Crossroad",
    3: "T_Junction",
    4: "Crossroad",
    5: "Other",
    6: "Roundabout",
    7: "Private_Access",
    8: "Other",
    9: "Unknown"
}


eth_junction_map = {
    "No junction": "No_Junction",
    "Y": "T_Junction",
    "T": "T_Junction",
    "+": "Crossroad",
    "X": "Crossroad",
    "O": "Other",
    "Other": "Other",
    "Out of junction": "No_Junction"
}


uk_collisions["junction_common"] = (
    uk_collisions["junction_detail"]
    .map(uk_junction_map)
)


fr_characteristics["junction_common"] = (
    fr_characteristics["int"]
    .map(fr_junction_map)
)


ethiopia["junction_common"] = (
    ethiopia["Types_of_Junction"]
    .str.strip()
    .map(eth_junction_map)
)


for name, data in {
    "UK": uk_collisions,
    "France": fr_characteristics,
    "Ethiopia": ethiopia
}.items():

    print(f"\n--- {name} ---")
    print(
        data["junction_common"]
        .value_counts(dropna=False)
    )
    print(
        "Missing:",
        data["junction_common"].isna().sum()
    )

In [ ]:
# Load vehicle tables for UK and France

uk_vehicles = pd.read_csv(
    "../data/raw/UK/dft-road-casualty-statistics-vehicle-2025.csv"
)

fr_vehicles = pd.read_csv(
    "../data/raw/France/vehicules-2024.csv",
    sep=";"
)

print("UK Vehicles:", uk_vehicles.shape)
print("France Vehicles:", fr_vehicles.shape)

In [ ]:
# بررسی نوع وسیله نقلیه در سه کشور قبل از هماهنگ‌سازی

print("--- UK vehicle type ---")
print(
    uk_vehicles["vehicle_type"]
    .value_counts(dropna=False)
    .head(30)
)

print("\n--- France vehicle type ---")
print(
    fr_vehicles["catv"]
    .value_counts(dropna=False)
)

print("\n--- Ethiopia vehicle type ---")
print(
    ethiopia["Type_of_vehicle"]
    .value_counts(dropna=False)
)

In [ ]:
# بررسی تعداد وسایل نقلیه در هر تصادف

uk_vehicle_count = (
    uk_vehicles
    .groupby("collision_index")
    .size()
)

fr_vehicle_count = (
    fr_vehicles
    .groupby("Num_Acc")
    .size()
)


print("--- UK vehicles per collision ---")
print(uk_vehicle_count.value_counts().head(10))

print("\n--- France vehicles per accident ---")
print(fr_vehicle_count.value_counts().head(10))

In [ ]:
# بررسی کدهای Vehicle Type در UK برای Mapping

uk_vehicle_codes = (
    uk_vehicles["vehicle_type"]
    .value_counts()
    .reset_index()
)

uk_vehicle_codes.columns = [
    "vehicle_type_code",
    "count"
]

print(uk_vehicle_codes)

In [ ]:
# بررسی کدهای Vehicle Type در France برای Mapping

fr_vehicle_codes = (
    fr_vehicles["catv"]
    .value_counts()
    .reset_index()
)

fr_vehicle_codes.columns = [
    "catv_code",
    "count"
]

print(fr_vehicle_codes)

In [ ]:
# استخراج کدهای UK vehicle type برای Mapping رسمی

uk_vehicle_codes = (
    uk_vehicles["vehicle_type"]
    .value_counts()
    .index
    .tolist()
)

print(uk_vehicle_codes)

In [ ]:
# بررسی Sheet های فایل راهنمای UK

uk_guide_path = "../data/raw/UK/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"

xls = pd.ExcelFile(uk_guide_path)

print(xls.sheet_names)

In [ ]:
# خواندن Sheet اصلی Code List

uk_code_list = pd.read_excel(
    uk_guide_path,
    sheet_name="2024_code_list"
)

print(uk_code_list.shape)
print(uk_code_list.head(10))

In [ ]:
# جستجوی کلمات مرتبط با Vehicle

vehicle_rows = uk_code_list[
    uk_code_list.astype(str)
    .apply(
        lambda row: row.str.contains(
            "vehicle",
            case=False,
            na=False
        ).any(),
        axis=1
    )
]

print(vehicle_rows.head(20))

In [ ]:
# استخراج کامل کدهای vehicle_type از Codebook UK

uk_vehicle_codebook = uk_code_list[
    (uk_code_list["table"] == "vehicle") &
    (uk_code_list["field name"] == "vehicle_type")
]

print(
    uk_vehicle_codebook[
        ["code/format", "label", "note"]
    ].to_string(index=False)
)

In [ ]:
# ساخت ویژگی‌های تجمیعی Vehicle در سطح Collision

uk_vehicle_category_map = {
    1: "two_wheeler",
    2: "motorcycle",
    3: "motorcycle",
    4: "motorcycle",
    5: "motorcycle",
    8: "taxi",
    9: "car",
    10: "public_transport",
    11: "public_transport",
    16: "special_vehicle",
    17: "special_vehicle",
    18: "special_vehicle",
    19: "light_commercial",
    20: "heavy_vehicle",
    21: "heavy_vehicle",
    22: "two_wheeler",
    23: "motorcycle",
    90: "other",
    97: "motorcycle",
    98: "heavy_vehicle",
    99: "unknown",
    103: "motorcycle",
    104: "motorcycle",
    105: "motorcycle",
    106: "motorcycle",
    108: "taxi",
    109: "car",
    110: "special_vehicle",
    113: "heavy_vehicle",
    -1: "unknown"
}

uk_vehicles["vehicle_category"] = (
    uk_vehicles["vehicle_type"]
    .map(uk_vehicle_category_map)
)

uk_vehicles["vehicle_category"].value_counts(dropna=False)

In [ ]:
uk_vehicle_features = (
    uk_vehicles
    .groupby("collision_index")
    .agg(
        vehicle_count=("vehicle_type","count"),
        has_motorcycle=("vehicle_category",
                        lambda x: int("motorcycle" in x.values)),
        has_heavy_vehicle=("vehicle_category",
                           lambda x: int("heavy_vehicle" in x.values)),
        has_public_transport=("vehicle_category",
                              lambda x: int("public_transport" in x.values)),
        has_two_wheeler=("vehicle_category",
                         lambda x: int("two_wheeler" in x.values))
    )
    .reset_index()
)

uk_vehicle_features.head()

In [ ]:
# بررسی تعداد تصادف‌های دارای Featureهای Vehicle

print("UK vehicle features shape:")
print(uk_vehicle_features.shape)

print("\nMissing values:")
print(uk_vehicle_features.isna().sum())

print("\nVehicle count distribution:")
print(
    uk_vehicle_features["vehicle_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
# France vehicle category mapping

fr_vehicle_category_map = {

    # Bicycle
    1: "two_wheeler",

    # Motorcycle / scooter
    2: "motorcycle",
    30: "motorcycle",
    31: "motorcycle",
    32: "motorcycle",
    33: "motorcycle",
    34: "motorcycle",

    # Passenger car
    7: "car",
    10: "car",

    # Heavy vehicles
    13: "heavy_vehicle",
    14: "heavy_vehicle",
    15: "heavy_vehicle",
    17: "heavy_vehicle",
    20: "heavy_vehicle",
    21: "heavy_vehicle",

    # Public transport
    37: "public_transport",
    38: "public_transport",

    # Two wheeler / personal mobility
    60: "two_wheeler",
    80: "two_wheeler",

    # Other categories
    50: "other",
    99: "unknown",
    0: "unknown",
    -1: "unknown"
}


fr_vehicles["vehicle_category"] = (
    fr_vehicles["catv"]
    .map(fr_vehicle_category_map)
)


print(
    fr_vehicles["vehicle_category"]
    .value_counts(dropna=False)
)

In [ ]:
fr_vehicle_features = (
    fr_vehicles
    .groupby("Num_Acc")
    .agg(
        vehicle_count=("catv","count"),

        has_motorcycle=("vehicle_category",
                        lambda x: int("motorcycle" in x.values)),

        has_heavy_vehicle=("vehicle_category",
                           lambda x: int("heavy_vehicle" in x.values)),

        has_public_transport=("vehicle_category",
                              lambda x: int("public_transport" in x.values)),

        has_two_wheeler=("vehicle_category",
                         lambda x: int("two_wheeler" in x.values))
    )
    .reset_index()
)


fr_vehicle_features.head()

In [ ]:
# بررسی کدهای catv که Mapping نشده‌اند

missing_vehicle_codes = (
    fr_vehicles[
        fr_vehicles["vehicle_category"].isna()
    ]["catv"]
    .value_counts()
)

print(missing_vehicle_codes)

In [ ]:
fr_vehicle_category_map.update({

    # Motorized two-wheelers / motorcycles
    3: "motorcycle",
    16: "motorcycle",
    35: "motorcycle",
    36: "motorcycle",
    39: "motorcycle",
    40: "motorcycle",
    41: "motorcycle",
    42: "motorcycle",
    43: "motorcycle"

})

In [ ]:
fr_vehicles["vehicle_category"] = (
    fr_vehicles["catv"]
    .map(fr_vehicle_category_map)
)


print(
    fr_vehicles["vehicle_category"]
    .value_counts(dropna=False)
)

In [ ]:
# ساخت دسته‌بندی مشترک Vehicle برای Ethiopia

eth_vehicle_category_map = {

    "Automobile": "car",
    "Stationwagen": "car",

    "Motorcycle": "motorcycle",
    "Bajaj": "motorcycle",
    "Turbo": "motorcycle",

    "Bicycle": "two_wheeler",

    "Lorry (41?100Q)": "heavy_vehicle",
    "Lorry (11?40Q)": "heavy_vehicle",
    "Long lorry": "heavy_vehicle",

    "Public (12 seats)": "public_transport",
    "Public (13?45 seats)": "public_transport",
    "Public (> 45 seats)": "public_transport",

    "Taxi": "taxi",

    "Special vehicle": "special_vehicle",
    "Ridden horse": "special_vehicle",

    "Other": "other"
}


ethiopia["vehicle_category"] = (
    ethiopia["Type_of_vehicle"]
    .map(eth_vehicle_category_map)
)


print(
    ethiopia["vehicle_category"]
    .value_counts(dropna=False)
)

In [ ]:
# ساخت Featureهای Vehicle سطح تصادف برای Ethiopia

ethiopia["vehicle_count"] = (
    ethiopia["Number_of_vehicles_involved"]
)

ethiopia["has_motorcycle"] = (
    ethiopia["vehicle_category"]
    .eq("motorcycle")
    .astype(int)
)

ethiopia["has_heavy_vehicle"] = (
    ethiopia["vehicle_category"]
    .eq("heavy_vehicle")
    .astype(int)
)

ethiopia["has_public_transport"] = (
    ethiopia["vehicle_category"]
    .eq("public_transport")
    .astype(int)
)

ethiopia["has_two_wheeler"] = (
    ethiopia["vehicle_category"]
    .eq("two_wheeler")
    .astype(int)
)

In [ ]:
# بررسی مقادیر Type_of_vehicle که Mapping نشده‌اند

missing_eth_vehicle = (
    ethiopia[
        ethiopia["vehicle_category"].isna()
    ]["Type_of_vehicle"]
    .value_counts(dropna=False)
)

print(missing_eth_vehicle)

In [ ]:
# بررسی دقیق Missing در Type_of_vehicle

print(
    ethiopia["Type_of_vehicle"]
    .isna()
    .sum()
)

print(
    (ethiopia["Type_of_vehicle"].astype(str).str.strip() == "")
    .sum()
)

In [ ]:
ethiopia[
    ethiopia["vehicle_category"].isna()
][
    ["Type_of_vehicle"]
].head(20)

In [ ]:
ethiopia["vehicle_category"] = (
    ethiopia["vehicle_category"]
    .fillna("unknown")
)

print(
    ethiopia["vehicle_category"]
    .value_counts()
)

In [ ]:
# ساخت ویژگی‌های Vehicle سطح تصادف برای Ethiopia

ethiopia["vehicle_count"] = (
    ethiopia["Number_of_vehicles_involved"]
)

ethiopia["has_motorcycle"] = (
    ethiopia["vehicle_category"]
    .eq("motorcycle")
    .astype(int)
)

ethiopia["has_heavy_vehicle"] = (
    ethiopia["vehicle_category"]
    .eq("heavy_vehicle")
    .astype(int)
)

ethiopia["has_public_transport"] = (
    ethiopia["vehicle_category"]
    .eq("public_transport")
    .astype(int)
)

ethiopia["has_two_wheeler"] = (
    ethiopia["vehicle_category"]
    .eq("two_wheeler")
    .astype(int)
)


ethiopia[
    [
        "vehicle_count",
        "has_motorcycle",
        "has_heavy_vehicle",
        "has_public_transport",
        "has_two_wheeler"
    ]
].head()

In [ ]:
print("Ethiopia vehicle features:")

print(
    ethiopia[
        [
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler"
        ]
    ].isna().sum()
)


print("\nVehicle count distribution:")

print(
    ethiopia["vehicle_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Merge UK collision data with aggregated vehicle features

uk_final = uk_collisions.merge(
    uk_vehicle_features,
    on="collision_index",
    how="left"
)

print("UK final shape:")
print(uk_final.shape)

print("\nMissing vehicle features:")
print(
    uk_final[
        [
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# Merge France characteristics with aggregated vehicle features

fr_final = fr_characteristics.merge(
    fr_vehicle_features,
    on="Num_Acc",
    how="left"
)

print("France final shape:")
print(fr_final.shape)

print("\nMissing vehicle features:")
print(
    fr_final[
        [
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
ethiopia_final = ethiopia.copy()

print(
    ethiopia_final.shape
)

In [ ]:
# بررسی Target فعلی UK

print(
    uk_final["severity_class"]
    .value_counts(dropna=False)
)

In [ ]:
# بررسی ستون های شدت در France final

print(
    fr_final[
        [
            col for col in fr_final.columns
            if "severity" in col.lower() or "grav" in col.lower()
        ]
    ].head()
)

print("\nColumns:")
print(
    [
        col for col in fr_final.columns
        if "severity" in col.lower() or "grav" in col.lower()
    ]
)

In [ ]:
# ساخت شدت تصادف در سطح Accident برای France

fr_severity = (
    fr_users
    .groupby("Num_Acc")["grav"]
    .max()
    .reset_index()
)

print(fr_severity.head())
print(fr_severity["grav"].value_counts())

In [ ]:
# اتصال شدت به دیتاست اصلی France

fr_final = fr_final.merge(
    fr_severity,
    on="Num_Acc",
    how="left"
)

print(fr_final.shape)
print(fr_final["grav"].isna().sum())

In [ ]:
fr_users["grav"].value_counts(dropna=False)

In [ ]:
# ساخت Severity در سطح Accident برای France

def france_severity(gravs):
    if 2 in gravs.values:
        return "Fatal"
    elif 3 in gravs.values:
        return "Serious"
    else:
        return "Slight"


fr_severity = (
    fr_users
    .groupby("Num_Acc")["grav"]
    .apply(france_severity)
    .reset_index(name="severity_class")
)


print(fr_severity.head())

print("\nSeverity distribution:")
print(
    fr_severity["severity_class"]
    .value_counts()
)

In [ ]:
# اضافه کردن Target به France final

fr_final = fr_final.merge(
    fr_severity,
    on="Num_Acc",
    how="left"
)


print(fr_final.shape)

print(
    fr_final["severity_class"]
    .isna()
    .sum()
)

In [ ]:
# Ethiopia severity harmonization

ethiopia_final["severity_class"] = (
    ethiopia_final["Accident_severity"]
    .map({
        "Slight Injury": "Slight",
        "Serious Injury": "Serious",
        "Fatal injury": "Fatal"
    })
)

print(
    ethiopia_final["severity_class"]
    .value_counts(dropna=False)
)

In [ ]:
# بررسی ستون های نهایی سه کشور

print("UK columns:")
print(uk_final.columns.tolist())

print("\nFrance columns:")
print(fr_final.columns.tolist())

print("\nEthiopia columns:")
print(ethiopia_final.columns.tolist())

In [ ]:
# بررسی ستون های مربوط به سطح جاده در France

print(
    [
        col for col in fr_locations.columns
        if "surf" in col.lower()
    ]
)

In [ ]:
final_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler",
    "severity_class"
]

In [ ]:
# ساخت surface feature در سطح تصادف برای France

fr_surface = (
    fr_locations
    .groupby("Num_Acc")["surface_common"]
    .first()
    .reset_index()
)

print(fr_surface.shape)
print(fr_surface.head())

In [ ]:
# Merge France road-surface feature into the final accident-level dataset

fr_final = fr_final.merge(
    fr_surface,
    on="Num_Acc",
    how="left"
)

print("France final shape after adding surface_common:")
print(fr_final.shape)

print("\nSurface columns:")
print([c for c in fr_final.columns if "surface" in c.lower()])

print("\nMissing surface_common:")
print(fr_final["surface_common"].isna().sum())


In [ ]:
# بررسی وضعیت فعلی دیتاست‌ها قبل از ساخت مدل

print("===== UK =====")
print("Shape:", uk_final.shape)
print(
    "Missing common features:",
    uk_final[
        [
            "weather_common",
            "light_common",
            "surface_common",
            "junction_common",
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler",
            "severity_class"
        ]
    ].isna().sum().sum()
)


print("\n===== France =====")
print("Shape:", fr_final.shape)

print(
    "Surface columns:",
    [c for c in fr_final.columns if "surface" in c.lower()]
)

print(
    "Missing common features:",
    fr_final[
        [
            "weather_common",
            "light_common",
            "surface_common",
            "junction_common",
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler",
            "severity_class"
        ]
    ].isna().sum().sum()
)


print("\n===== Ethiopia =====")
print("Shape:", ethiopia_final.shape)

print(
    "Missing common features:",
    ethiopia_final[
        [
            "weather_common",
            "light_common",
            "surface_common",
            "junction_common",
            "vehicle_count",
            "has_motorcycle",
            "has_heavy_vehicle",
            "has_public_transport",
            "has_two_wheeler",
            "severity_class"
        ]
    ].isna().sum().sum()
)

In [ ]:
# تعریف Featureهای نهایی مشترک بین سه کشور

final_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler",
    "severity_class"
]

print(final_features)

In [ ]:
uk_model = uk_final[final_features].copy()
uk_model["country"] = "UK"

print(uk_model.shape)

In [ ]:
fr_model = fr_final[final_features].copy()
fr_model["country"] = "France"

print(fr_model.shape)

In [ ]:
eth_model = ethiopia_final[final_features].copy()
eth_model["country"] = "Ethiopia"

print(eth_model.shape)

In [ ]:
cross_country_dataset = pd.concat(
    [
        uk_model,
        fr_model,
        eth_model
    ],
    ignore_index=True
)

In [ ]:
uk_final.to_csv(
    "../data/processed/uk_processed.csv",
    index=False
)

In [ ]:
fr_final.to_csv(
    "../data/processed/france_processed.csv",
    index=False
)

In [ ]:
ethiopia_final.to_csv(
    "../data/processed/ethiopia_processed.csv",
    index=False
)

In [ ]:
import pandas as pd

dataset_summary = pd.DataFrame({
    "Country": [
        "UK",
        "France",
        "Ethiopia"
    ],
    "Records": [
        uk_final.shape[0],
        fr_final.shape[0],
        ethiopia_final.shape[0]
    ],
    "Columns": [
        uk_final.shape[1],
        fr_final.shape[1],
        ethiopia_final.shape[1]
    ],
    "Target": [
        "severity_class",
        "severity_class",
        "severity_class"
    ]
})

dataset_summary.to_csv(
    "../outputs/tables/dataset_summary.csv",
    index=False
)

dataset_summary

In [ ]:
severity_distribution = pd.DataFrame({
    "UK": uk_final["severity_class"].value_counts(),
    "France": fr_final["severity_class"].value_counts(),
    "Ethiopia": ethiopia_final["severity_class"].value_counts()
})

severity_distribution = (
    severity_distribution
    .fillna(0)
    .astype(int)
)

severity_distribution.to_csv(
    "../outputs/tables/severity_distribution.csv"
)

severity_distribution

In [ ]:
feature_map = pd.DataFrame({

    "Feature": [
        "hour",
        "day_common",
        "weather_common",
        "light_common",
        "surface_common",
        "junction_common",
        "vehicle_count",
        "has_motorcycle",
        "has_heavy_vehicle",
        "has_public_transport",
        "has_two_wheeler",
        "severity_class"
    ],

    "Description": [
        "Hour of accident occurrence",
        "Day of week",
        "Weather condition",
        "Lighting condition",
        "Road surface condition",
        "Junction type",
        "Number of involved vehicles",
        "Presence of motorcycle",
        "Presence of heavy vehicle",
        "Presence of public transport",
        "Presence of two wheeler",
        "Accident severity"
    ]
})


feature_map.to_csv(
    "../outputs/tables/feature_harmonization_map.csv",
    index=False
)

feature_map

In [ ]:
validation = pd.DataFrame({

    "Country": [
        "UK",
        "France",
        "Ethiopia"
    ],

    "Missing_Common_Features": [
        0,
        0,
        0
    ],

    "Vehicle_Features_Status": [
        "Completed",
        "Completed",
        "Completed"
    ],

    "Severity_Harmonization": [
        "Completed",
        "Completed",
        "Completed"
    ]
})


validation.to_csv(
    "../outputs/tables/preprocessing_validation.csv",
    index=False
)

validation

In [ ]:
# پیدا کردن ستون های مشترک سه کشور

uk_cols = set(uk_final.columns)
fr_cols = set(fr_final.columns)
eth_cols = set(ethiopia_final.columns)

common_features = (
    uk_cols
    .intersection(fr_cols)
    .intersection(eth_cols)
)

print("Common features:")
for c in sorted(common_features):
    print(c)

print("\nNumber of common features:", len(common_features))

In [ ]:
feature_matrix = pd.DataFrame({
    "Feature": [
        "hour",
        "day_common",
        "weather_common",
        "light_common",
        "surface_common",
        "junction_common",
        "vehicle_count",
        "has_motorcycle",
        "has_heavy_vehicle",
        "has_public_transport",
        "has_two_wheeler",
        "severity_class"
    ],
    "UK": "Yes",
    "France": "Yes",
    "Ethiopia": "Yes",
    "Used_in_Cross_Country_Model": "Yes"
})


feature_matrix.to_csv(
    "../outputs/tables/feature_compatibility_matrix.csv",
    index=False
)

feature_matrix

In [ ]:
# Featureهای مشترک نهایی برای مدل بین‌کشوری

final_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler",
    "severity_class"
]

print("Number of features:", len(final_features))

In [ ]:
uk_model = uk_final[final_features].copy()

uk_model["country"] = "UK"

print("UK model shape:")
print(uk_model.shape)

In [ ]:
fr_model = fr_final[final_features].copy()

fr_model["country"] = "France"

print("France model shape:")
print(fr_model.shape)

In [ ]:
eth_model = ethiopia_final[final_features].copy()

eth_model["country"] = "Ethiopia"

print("Ethiopia model shape:")
print(eth_model.shape)

In [ ]:
# بررسی یکسان بودن ساختار سه دیتاست

print(
    uk_model.columns.tolist()
)

print(
    uk_model.columns.equals(fr_model.columns)
)

print(
    uk_model.columns.equals(eth_model.columns)
)

In [ ]:
# ساخت دیتاست نهایی Cross-Country

cross_country_dataset = pd.concat(
    [
        uk_model,
        fr_model,
        eth_model
    ],
    ignore_index=True
)


print("Cross-country dataset shape:")
print(cross_country_dataset.shape)


print("\nCountry distribution:")
print(
    cross_country_dataset["country"]
    .value_counts()
)


print("\nSeverity distribution:")
print(
    cross_country_dataset["severity_class"]
    .value_counts()
)

In [ ]:
print(
    cross_country_dataset.isna().sum()
)

In [ ]:
cross_country_dataset.to_csv(
    "../data/processed/cross_country_dataset.csv",
    index=False
)

print("Saved successfully")

In [ ]:
cross_summary = pd.DataFrame({
    "Country": 
        cross_country_dataset["country"].value_counts().index,
    
    "Records":
        cross_country_dataset["country"].value_counts().values
})

cross_summary.to_csv(
    "../outputs/tables/cross_country_dataset_summary.csv",
    index=False
)

cross_summary